# List-aware · broader individual-model and feature-pool screen

**Stage A before the list-aware ensemble factory.** The new `curated` suite carries forward promising features from the 1,050-home individual and composite screens without claiming those results generalize. Compare **a curated three-family feature-recipe suite (or the original 44-model catalog)** (the entire catalog) in a factorized research screen. The goal is a **diverse candidate pool**, *not* choosing an ensemble or freezing any feature group yet.

The experiment distinguishes:

- Model families / objectives: CatBoost MAE, Huber, RMSE; tuned XGBoost and LightGBM, direct or sale/list ratio targets.
- Text: no text, original TF-IDF, selected binary, training-fitted full binary/count/TF-IDF.
- Comparable sales: original, age-first, same-property-type quota, both.
- Spatial additions: splines alone, geography alone, both, and prior AVM designs.

**Screen:** Mar–May 2025, 10,000-row/350-home/300-tree pilot defaults, monthly expanding origins.
**Confirmation:** Jun–Jul 2025, full legal history/complete months/original tree counts, ONLY the top capped shortlist chosen from the screen.

**Important:** historical months have been studied previously; these are exploratory development results. Do **not** compare pilot MdAPE directly with the earlier **3.8996%** full-strength single-fit fixed-origin Aug–Oct reference—different months, fits and budgets.

The notebook delegates fitting and caching to `comparisons/listaware_pool_screen.py` and **does not** change the production models or off-market factory.


## 1 · Find the checkout and load the experiment catalog

Works from `experiments/` or the repository root. You may set `DATA_DASH_ROOT` to the checkout. Keep your listings Parquet/CSV source unchanged between runs.


In [ ]:
from pathlib import Path
import os
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

start = Path(os.environ.get("DATA_DASH_ROOT") or Path.cwd()).expanduser().resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "feature" / "listaware_pool_screen.py").is_file()
             and (p / "data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the repository or set DATA_DASH_ROOT.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from comparisons.listaware_pool_screen import (
    CATALOG, GROUPS, SUITES, SCREEN_MONTHS, CONFIRM_MONTHS,
    SEED_REFERENCE, catalog_frame, choose_confirmation, run,
)
print("Repository:", ROOT)
print("Candidates in full catalog:", len(CATALOG))
display(pd.DataFrame([
    {"suite": suite, "members": len(names)} for suite, names in SUITES.items()
]))


Repository: C:\Users\deant\Desktop\Coding\Data Dash Case Comp
Candidates in full catalog: 44


,suite,members
0,core,8
1,text,15
2,comparables,11
3,spatial,12
4,objectives,8
5,avm,10
6,all,44


## 2 · Configure the screen and later confirmation

Start with `SUITE="core"` for a quick pipeline check, then use `"curated"` for the pooled-feature comparison. The previous `"all"` catalog remains available and unchanged. Use different output directories across suites. Full-budget selection then uses a 14-member shortlist with predeclared family/feature anchors.

`RUN_SCREEN` and `RUN_CONFIRM` default to **False**, so opening or executing this notebook does not unexpectedly train dozens of models. Rerun with identical settings to resume cached candidate/month predictions.


In [3]:
SUITE = "curated"    # curated | core | text | comparables | spatial | objectives | avm | all
THREADS = 4
SEED = 42
BOOTSTRAP_REPEATS = 300

SCREEN_TRAIN_CAP = 10_000
SCREEN_VALIDATION_CAP = 350
SCREEN_TREE_CAP = 300

TOP_K = 8  # screen-ranked pool with mandatory old controls + family diversity
RUN_TESTS = False
RUN_SCREEN = False
RUN_CONFIRM = False

SCREEN_DIR = ROOT / "outputs" / f"listaware_pool_screen_{SUITE}"
CONFIRM_DIR = ROOT / "outputs" / f"listaware_pool_confirmation_{SUITE}"

print("Screen:", SCREEN_DIR)
print("Confirmation:", CONFIRM_DIR)
print("Pilot folds:", SCREEN_MONTHS)
print("Later full-strength folds:", CONFIRM_MONTHS)
print("Candidates:", len(SUITES[SUITE]))


Screen: C:\Users\deant\Desktop\Coding\Data Dash Case Comp\outputs\listaware_pool_screen_all
Confirmation: C:\Users\deant\Desktop\Coding\Data Dash Case Comp\outputs\listaware_pool_confirmation_all
Pilot folds: ('2025-03', '2025-04', '2025-05')
Later full-strength folds: ('2025-06', '2025-07')
Candidates: 44


## 3 · Candidate catalog

Every screen compares the same held-out homes and includes `cat_huber_baseline` as a matched reference. All candidates use the existing list-aware data-hiding rules; the original 50/35/15 blend is **not** refitted or reweighted in this individual-model screen.

There is deliberately **no** feature bagging, row bootstrapping or Optuna here: first identify which feature/model configurations merit becoming ingredients of those next-stage algorithms.


In [4]:
candidate_catalog = catalog_frame(SUITES[SUITE])
display(candidate_catalog)
display(candidate_catalog.groupby(["family", "target"]).size().rename("candidate_count"))


,variant,family,loss,target,features,tuned_parameters
0,cat_huber_baseline,catboost,Huber:delta=0.05,ratio,Xpt_spatiotemporal,False
1,cat_mae,catboost,MAE,ratio,Xpt_spatiotemporal,False
2,cat_rmse,catboost,RMSE,ratio,Xpt_spatiotemporal,False
3,cat_tuned_rmse,catboost,RMSE,ratio,Xpt_spatiotemporal,True
4,xgb_tuned_binary,xgboost,RMSE,ratio,Xpt_spatiotemporal__selected_binary,True
5,xgb_spline_geo,xgboost,RMSE,ratio,Xpt_spatiotemporal__selected_binary__st_spacet...,True
6,lgb_tuned_binary,lightgbm,RMSE,ratio,Xpt_spatiotemporal__selected_binary,True
7,lgb_spline_geo,lightgbm,RMSE,ratio,Xpt_spatiotemporal__selected_binary__st_spacet...,True
8,cat_mae_no_text,catboost,MAE,ratio,Xpt_spatiotemporal__no_text,False
9,cat_mae_binary,catboost,MAE,ratio,Xpt_spatiotemporal__selected_binary,False


family    target
catboost  direct     2
          ratio     18
lightgbm  direct     1
          ratio     11
xgboost   direct     1
          ratio     11
Name: candidate_count, dtype: int64

## 4 · Optional fast safety tests

The tests use synthetic listings and stub model fits to check candidate registration, exact month/home matching, no answer leakage, strict cache manifests, resume, and deterministic shortlist selection. They do not run a full housing model.


In [5]:
if RUN_TESTS:
    import subprocess
    import tempfile
    # Each run owns fresh temporary/cache folders, avoiding AppData permissions.
    test_runs = ROOT / "outputs" / "test_runs"
    test_runs.mkdir(parents=True, exist_ok=True)
    test_run = Path(tempfile.mkdtemp(prefix="listaware-", dir=test_runs))
    test_result = subprocess.run([
        sys.executable, "-m", "pytest",
        "tests/test_listaware_pool_screen.py",
        "tests/test_listaware_shotgun.py",
        "-q",
        "--basetemp", str(test_run / "tmp"),
        "-o", f"cache_dir={test_run / 'cache'}",
    ], cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
       text=True, encoding="utf-8", errors="replace")
    test_log = test_run / "pytest.log"
    test_log.write_text(test_result.stdout, encoding="utf-8")
    print(test_result.stdout, end="", flush=True)
    print(f"Full test output: {test_log}", flush=True)
    test_result.check_returncode()
else:
    print("Tests skipped; set RUN_TESTS=True to validate the setup.")


................                                                         [100%]
16 passed in 7.74s
Full test output: C:\Users\deant\Desktop\Coding\Data Dash Case Comp\outputs\test_runs\listaware-qwre4xma\pytest.log


## 5 · Run or resume the bounded chronological screen (March–May)

`RUN_SCREEN=True` fits each candidate on sale histories that end **before that month's first day**, then validates on that month's sales with sold price and post-sale fields hidden.

Outputs and each completed model-month forecast are written incrementally; rerun after interruption with **unchanged settings**, not a new candidate set in the same output directory.


In [6]:
screen_pooled = screen_monthly = screen_types = screen_predictions = None

if RUN_SCREEN:
    screen_pooled, screen_monthly, screen_types, screen_predictions = run(
        stage="screen", suite=SUITE, output_dir=SCREEN_DIR,
        train_cap=SCREEN_TRAIN_CAP, validation_cap=SCREEN_VALIDATION_CAP,
        tree_cap=SCREEN_TREE_CAP, threads=THREADS, seed=SEED,
        bootstrap_repeats=BOOTSTRAP_REPEATS,
    )
elif (SCREEN_DIR / "pooled_scores.csv").is_file():
    screen_record = json.loads((SCREEN_DIR / "manifest.json").read_text())
    if (screen_record.get("stage") != "screen"
        or set(screen_record.get("candidates", ())) != set(SUITES[SUITE])
        or tuple(screen_record.get("months", ())) != SCREEN_MONTHS):
        raise ValueError("This screen directory belongs to a different protocol.")
    screen_pooled = pd.read_csv(SCREEN_DIR / "pooled_scores.csv")
    screen_monthly = pd.read_csv(SCREEN_DIR / "monthly_scores.csv")
    screen_types = pd.read_csv(SCREEN_DIR / "house_type_scores.csv")
    print("Loaded saved screen reports; no training.")
else:
    print("No finished screen; set RUN_SCREEN=True in section 2.")


[screen 2025-03 1/44] cat_huber_baseline: verified cached predictions
[screen 2025-03 2/44] cat_mae: verified cached predictions
[screen 2025-03 3/44] cat_rmse: verified cached predictions
[screen 2025-03 4/44] cat_tuned_rmse: verified cached predictions
[screen 2025-03 5/44] xgb_tuned_binary: verified cached predictions
[screen 2025-03 6/44] xgb_spline_geo: verified cached predictions
[screen 2025-03 7/44] lgb_tuned_binary: verified cached predictions
[screen 2025-03 8/44] lgb_spline_geo: verified cached predictions
[screen 2025-03 9/44] cat_mae_no_text: verified cached predictions
[screen 2025-03 10/44] cat_mae_binary: verified cached predictions
[screen 2025-03 11/44] cat_mae_full_binary: verified cached predictions
[screen 2025-03 12/44] cat_mae_full_count: verified cached predictions
[screen 2025-03 13/44] cat_mae_full_tfidf: verified cached predictions
[screen 2025-03 14/44] xgb_no_text: verified cached predictions
[screen 2025-03 15/44] xgb_full_binary: verified cached predictio

## 6 · Inspect individual model strength and error stability

`delta_mdape_pp` uses a matched CatBoost Huber reference **on the same screen folds**, not the earlier 3.8996% final-stage blend. Negative deltas favour the challenger.

Compare MdAPE, MAPE, P90 APE and all monthly folds. A model can be retained later for diversity even if it is not the best standalone performer.


In [7]:
if screen_pooled is not None:
    screen_pooled = screen_pooled.copy()
    screen_pooled["screen_objective"] = .65*screen_pooled.mdape + .35*screen_pooled.mape
    view = screen_pooled.sort_values(["screen_objective", "mdape"])
    display(view[[
        "variant", "homes", "mdape", "mape", "p90_ape",
        "delta_mdape_pp", "delta_mape_pp", "screen_objective",
    ]].round(4).reset_index(drop=True))

    display(Markdown("**Monthly MdAPE (lower is better)**"))
    display(screen_monthly.pivot_table(
        index="variant", columns="month", values="mdape"
    ).reindex(view.variant).round(4))
else:
    print("Run or load screen results first.")


,variant,homes,mdape,mape,p90_ape,delta_mdape_pp,delta_mape_pp,screen_objective
0,cat_mae_spatial_geo,1050,4.6685,6.1159,13.0367,-0.0707,-0.0968,5.1751
1,cat_mae_binary,1050,4.6941,6.1421,13.2532,-0.0452,-0.0705,5.2009
2,cat_mae_spatial_geo_type_quota,1050,4.7131,6.1191,12.9986,-0.0261,-0.0935,5.2052
3,cat_mae_splines,1050,4.7161,6.1304,13.1650,-0.0231,-0.0823,5.2111
4,cat_huber_adjusted,1050,4.7225,6.1392,13.1141,-0.0168,-0.0734,5.2183
5,xgb_splines,1050,4.7444,6.1512,13.3886,0.0052,-0.0614,5.2368
6,cat_huber_advanced,1050,4.7600,6.1423,13.0581,0.0208,-0.0703,5.2438
7,cat_mae_geo,1050,4.7545,6.1561,13.1506,0.0153,-0.0566,5.2451
8,cat_huber_baseline,1050,4.7392,6.2126,13.2944,0.0000,0.0000,5.2549
9,cat_mae_full_binary,1050,4.7694,6.1860,13.3031,0.0302,-0.0266,5.2652


**Monthly MdAPE (lower is better)**

month,2025-03,2025-04,2025-05
variant,,,
cat_mae_spatial_geo,4.6461,4.4888,4.8377
cat_mae_binary,4.5520,4.7016,4.8706
cat_mae_spatial_geo_type_quota,4.5112,4.6548,5.0269
cat_mae_splines,4.4948,4.4998,4.9229
cat_huber_adjusted,4.5960,4.4083,4.9963
xgb_splines,4.8794,4.4776,4.9307
cat_huber_advanced,4.7539,4.6791,4.8668
cat_mae_geo,4.7682,4.5457,4.9178
cat_huber_baseline,4.6967,4.5778,4.8347


In [11]:
from pathlib import Path
from dataclasses import asdict
from itertools import product
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_FAMILY = "catboost"       # catboost | xgboost | lightgbm
LOSS = "MAE"                    # Used for CatBoost
TUNED = False                   # Existing tuned XGB/LGB parameters if applicable

MONTHS = ("2025-03", "2025-04", "2025-05")
TRAIN_CAP = 10_000
VALIDATION_CAP = 350
TREE_CAP = 300
THREADS = 4
SEED = 42

# Only ONE residual model may be selected.
# None | knn | rbf_ridge | tensor_gam | spline_gam | local_kernel
RESIDUAL_METHOD = None
RESIDUAL_WEIGHT = 0.5
CALIBRATION_CAP = 350

BOOTSTRAP_REPEATS = 300

OUTPUT_DIR = Path(
    f"outputs/listaware_composite_{MODEL_FAMILY}_{LOSS.lower()}"
)

# ============================================================
# 2. IMPORT EXISTING PROJECT CODE
# ============================================================

if "ROOT" in globals():
    os.chdir(ROOT)

import data

from features.features import make_features
from models.listaware_shotgun_specs import Candidate
from models.spacetime_screen_residuals import SpaceTimeResidual

from listaware_quick_experiment import (
    _fit_predict,
    _frame,
    _verified_cache,
    _atomic_csv,
    _dataset_signature,
)

from spacetime_fast import _cap

if MODEL_FAMILY not in {"catboost", "xgboost", "lightgbm"}:
    raise ValueError("Select exactly one model family")

if RESIDUAL_METHOD is not None:
    from models.spacetime_screen_residuals import RESIDUAL_VARIANTS
    if RESIDUAL_METHOD not in RESIDUAL_VARIANTS:
        raise ValueError("Unknown residual model")

# ============================================================
# 3. BUILD COMPATIBLE FEATURE COMBINATIONS
# ============================================================

# Exactly one text representation per candidate.
TEXT_OPTIONS = (
    "legacy",          # Original selected TF-IDF representation
    "selected_binary", # Original selected binary BoW
)

# Exactly one comparable-selection strategy per candidate.
COMP_OPTIONS = (
    "base",
    "comp_type_quota",
)

# Multiple independent spatial additions can coexist.
SPATIAL_OPTIONS = (
    (),
    ("spacetime_splines",),
    ("geo_context",),
    ("spacetime_splines", "geo_context"),
)

recipes = list(product(
    TEXT_OPTIONS,
    COMP_OPTIONS,
    SPATIAL_OPTIONS,
))

# Extra combinations motivated by the previous leaderboard.
recipes += [
    ("full_binary", "base",
     ("spacetime_splines", "geo_context")),

    ("full_binary", "comp_type_quota",
     ("spacetime_splines", "geo_context")),

    ("selected_binary", "comp_age_first",
     ("spacetime_splines", "geo_context")),

    ("selected_binary", "comp_both",
     ("spacetime_splines", "geo_context")),

    ("selected_binary", "base",
     ("spacetime_splines", "geo_context", "rw_adjusted_comps")),

    ("selected_binary", "comp_type_quota",
     ("spacetime_splines", "geo_context", "rw_adjusted_comps")),
]

def build_feature_name(text, comp, spatial):
    name = "Xpt_spatiotemporal"

    if text != "legacy":
        name += f"__{text}"

    if comp != "base":
        name += f"__{comp}"

    if spatial:
        name += "__st_" + "+".join(spatial)

    return name


def build_candidate(text, comp, spatial):
    spatial_label = "+".join(spatial) if spatial else "none"

    name = f"{MODEL_FAMILY}_{text}_{comp}_{spatial_label}"

    spec = Candidate(
        name=name,
        backend=MODEL_FAMILY,
        feature_name=build_feature_name(text, comp, spatial),
        target="ratio",
        loss=LOSS if MODEL_FAMILY == "catboost" else "RMSE",
        tuned=TUNED,
    )
    return name, spec


CANDIDATES = dict(build_candidate(*recipe) for recipe in recipes)

# Validate feature compatibility BEFORE expensive training.
for name, spec in CANDIDATES.items():
    try:
        make_features(
            spec.feature_name,
            model=MODEL_FAMILY,
            track="list_aware",
        )
    except Exception as exc:
        raise ValueError(
            f"Incompatible features for {name}: "
            f"{spec.feature_name}"
        ) from exc

REFERENCE = f"{MODEL_FAMILY}_selected_binary_base_spacetime_splines+geo_context"

assert REFERENCE in CANDIDATES

print("Model family:", MODEL_FAMILY)
print("Candidates:", len(CANDIDATES))
print("Residual correction:", RESIDUAL_METHOD)

display(pd.DataFrame([
    {
        "candidate": name,
        "features": spec.feature_name,
        "model": spec.backend,
        "loss": spec.loss,
    }
    for name, spec in CANDIDATES.items()
]))

# ============================================================
# 4. LOAD DATA AND VALIDATE EXPERIMENT MANIFEST
# ============================================================

sales = data.load("listings").copy()
sales["date_sold"] = pd.to_datetime(sales["date_sold"])
sales["date_listed"] = pd.to_datetime(sales["date_listed"])

if sales.ml_num.astype(str).duplicated().any():
    raise ValueError("Duplicate MLS identifiers")

if (
    not np.isfinite(sales[["sold_price", "list_price"]].to_numpy(float)).all()
    or (sales[["sold_price", "list_price"]] <= 0).any().any()
):
    raise ValueError("Invalid historical prices")

manifest = {
    "version": 1,
    "stage": "listaware_composite_screen",
    "model_family": MODEL_FAMILY,
    "loss": LOSS,
    "tuned": TUNED,
    "months": list(MONTHS),
    "train_cap": TRAIN_CAP,
    "validation_cap": VALIDATION_CAP,
    "tree_cap": TREE_CAP,
    "threads": THREADS,
    "seed": SEED,
    "residual_method": RESIDUAL_METHOD,
    "residual_weight": RESIDUAL_WEIGHT,
    "calibration_cap": CALIBRATION_CAP,
    "bootstrap_repeats": BOOTSTRAP_REPEATS,
    "data_sha256": _dataset_signature(sales),
    "candidates": {
        name: asdict(spec) for name, spec in CANDIDATES.items()
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_DIR / "manifest.json"

if manifest_path.exists():
    previous = json.loads(manifest_path.read_text())
    if previous != manifest:
        raise ValueError(
            "Experiment settings changed. Use a NEW OUTPUT_DIR."
        )
else:
    if any(OUTPUT_DIR.iterdir()):
        raise ValueError("Output directory is nonempty without a manifest")
    manifest_path.write_text(json.dumps(manifest, indent=2))

# ============================================================
# 5. CHRONOLOGICAL TRAINING AND PREDICTION CACHE
# ============================================================

sale_months = sales.date_sold.dt.to_period("M")

def get_cohort(month, validation_cap):
    period = pd.Period(month, freq="M")
    cutoff = period.start_time

    train = _cap(
        sales.loc[sales.date_sold < cutoff],
        TRAIN_CAP,
        SEED,
    )

    held = _cap(
        sales.loc[sale_months == period],
        validation_cap,
        SEED + 11,
    )

    if train.empty or held.empty:
        raise ValueError(f"Empty training/validation cohort: {month}")

    if not (train.date_sold < cutoff).all():
        raise ValueError("Future sale labels entered training")

    return period, train, held


def predict_candidate(name, spec, month, *, calibration=False):
    cap = CALIBRATION_CAP if calibration else VALIDATION_CAP

    period, train, held = get_cohort(month, cap)

    kind = "calibration" if calibration else "model"
    folder = "calibration" if calibration else "models"

    path = OUTPUT_DIR / folder / month / f"{name}.csv"

    cached = _verified_cache(
        path, held, period, kind, name
    )

    if cached is not None:
        return cached, held

    hidden = data.hide_answers(held, "list_aware")

    assert "sold_price" not in hidden
    assert "date_sold" not in hidden
    assert "list_price" in hidden

    predictions = _fit_predict(
        spec,
        train,
        hidden,
        tree_cap=TREE_CAP,
        threads=THREADS,
        features_cache={},
    )

    result = _frame(
        held, period, kind, name, predictions
    )

    _atomic_csv(path, result)

    return result, held


# ============================================================
# 6. OPTIONAL OUT-OF-TIME RESIDUAL CORRECTION
# ============================================================

def correct_with_residual(name, spec, month, base_result, held):
    """
    Calibrate on predictions made strictly before the target month.
    Never use in-sample training residuals or target-month labels.

    One residual model is fitted for each candidate and target month.
    """

    if RESIDUAL_METHOD is None:
        return None

    period = pd.Period(month, freq="M")
    label = f"{name}__resid_{RESIDUAL_METHOD}"

    path = OUTPUT_DIR / "residual" / month / f"{label}.csv"

    cached = _verified_cache(
        path, held, period, "residual", label
    )
    if cached is not None:
        return cached

    calibration_homes = []
    calibration_errors = []

    # Two earlier completed-sale months.
    for offset in (2, 1):
        earlier = str(period - offset)

        previous_result, previous_held = predict_candidate(
            name, spec, earlier, calibration=True
        )

        # These labels were completed before the target origin.
        if not (
            pd.to_datetime(previous_held.date_sold)
            < period.start_time
        ).all():
            raise ValueError("Residual calibration leaked future labels")

        y = previous_result.sold_price.to_numpy(float)
        p = previous_result.predicted_price.to_numpy(float)

        calibration_homes.append(previous_held)
        calibration_errors.append(np.log(y) - np.log(p))

    calibrator = SpaceTimeResidual(
        RESIDUAL_METHOD, seed=SEED
    )

    calibrator.fit(
        pd.concat(calibration_homes, ignore_index=True),
        np.concatenate(calibration_errors),
    )

    hidden = data.hide_answers(held, "list_aware")

    correction = calibrator.predict(hidden)

    # Conservative, fixed correction strength.
    adjusted = (
        base_result.predicted_price.to_numpy(float)
        * np.exp(RESIDUAL_WEIGHT * correction)
    )

    if not np.isfinite(adjusted).all() or (adjusted <= 0).any():
        raise ValueError("Invalid residual-adjusted prices")

    result = _frame(
        held, period, "residual", label, adjusted
    )
    _atomic_csv(path, result)

    return result


# ============================================================
# 7. RUN / RESUME ALL COMBINATIONS
# ============================================================

frames = []

for i, (name, spec) in enumerate(CANDIDATES.items(), 1):

    print(
        f"\n[{i}/{len(CANDIDATES)}] {name}",
        flush=True
    )

    for month in MONTHS:
        raw, held = predict_candidate(name, spec, month)
        frames.append(raw)

        if RESIDUAL_METHOD is not None:
            corrected = correct_with_residual(
                name, spec, month, raw, held
            )
            frames.append(corrected)

all_predictions = pd.concat(frames, ignore_index=True)

_atomic_csv(
    OUTPUT_DIR / "predictions.csv",
    all_predictions,
)

# ============================================================
# 8. MATCHED-HOME LEADERBOARD
# ============================================================

reference = (
    all_predictions.loc[
        all_predictions.variant.eq(REFERENCE)
    ]
    .set_index(["month", "ml_num"])
    .sort_index()
)

if reference.empty or not reference.index.is_unique:
    raise ValueError("Missing or duplicated reference model")

actual = reference.sold_price.to_numpy(float)
reference_pred = reference.predicted_price.to_numpy(float)
reference_ape = 100 * np.abs(reference_pred - actual) / actual
reference_log_error = np.log(actual) - np.log(reference_pred)

rng = np.random.default_rng(SEED)
n = len(actual)

# Shared paired bootstrap resamples across all candidates.
bootstrap_indices = rng.integers(
    0, n, size=(BOOTSTRAP_REPEATS, n)
)

rows = []
monthly_rows = []

for (arm, name), group in all_predictions.groupby(
    ["arm", "variant"], sort=False
):
    aligned = group.set_index(["month", "ml_num"]).sort_index()

    if (
        len(aligned) != len(reference)
        or not aligned.index.equals(reference.index)
        or not np.allclose(
            aligned.sold_price.to_numpy(float),
            actual,
            rtol=0,
            atol=0.0001,
        )
    ):
        raise ValueError(f"Unmatched evaluation cohort: {name}")

    pred = aligned.predicted_price.to_numpy(float)
    ape = 100 * np.abs(pred - actual) / actual

    mdape = float(np.median(ape))
    mape = float(np.mean(ape))
    p90 = float(np.quantile(ape, 0.9))

    bootstrap_deltas = (
        np.median(ape[bootstrap_indices], axis=1)
        - np.median(reference_ape[bootstrap_indices], axis=1)
    )

    candidate_log_error = np.log(actual) - np.log(pred)

    correlation = (
        float(np.corrcoef(
            reference_log_error,
            candidate_log_error
        )[0, 1])
        if np.std(candidate_log_error) > 0
        and np.std(reference_log_error) > 0
        else np.nan
    )

    rows.append({
        "arm": arm,
        "variant": name,
        "homes": n,
        "mdape": mdape,
        "mape": mape,
        "p90_ape": p90,
        "delta_mdape_pp": mdape - np.median(reference_ape),
        "delta_mape_pp": mape - np.mean(reference_ape),
        "ci_low": float(np.quantile(bootstrap_deltas, 0.025)),
        "ci_high": float(np.quantile(bootstrap_deltas, 0.975)),
        "fraction_homes_improved": float(np.mean(ape < reference_ape)),
        "log_error_correlation": correlation,
    })

    for month in MONTHS:
        mask = (
            aligned.index.get_level_values("month") == month
        )

        monthly_rows.append({
            "month": month,
            "variant": name,
            "mdape": float(np.median(ape[mask])),
            "mape": float(np.mean(ape[mask])),
            "delta_mdape_pp": float(
                np.median(ape[mask])
                - np.median(reference_ape[mask])
            ),
        })

leaderboard = (
    pd.DataFrame(rows)
    .sort_values(["mdape", "mape"])
    .reset_index(drop=True)
)

monthly_scores = pd.DataFrame(monthly_rows)

_atomic_csv(
    OUTPUT_DIR / "leaderboard.csv",
    leaderboard,
)
_atomic_csv(
    OUTPUT_DIR / "monthly_scores.csv",
    monthly_scores,
)

print("\nCOMPOSITE FEATURE SCREEN")
print("Reference:", REFERENCE)
print("Negative delta = better than reference on matched homes.")

display(leaderboard.round(4))

print("\nMONTHLY MdAPE")
display(
    monthly_scores.pivot(
        index="variant",
        columns="month",
        values="mdape",
    ).round(4)
)

print("\nSaved to:", OUTPUT_DIR.resolve())

Model family: catboost
Candidates: 22
Residual correction: None


,candidate,features,model,loss
0,catboost_legacy_base_none,Xpt_spatiotemporal,catboost,MAE
1,catboost_legacy_base_spacetime_splines,Xpt_spatiotemporal__st_spacetime_splines,catboost,MAE
2,catboost_legacy_base_geo_context,Xpt_spatiotemporal__st_geo_context,catboost,MAE
3,catboost_legacy_base_spacetime_splines+geo_con...,Xpt_spatiotemporal__st_spacetime_splines+geo_c...,catboost,MAE
4,catboost_legacy_comp_type_quota_none,Xpt_spatiotemporal__comp_type_quota,catboost,MAE
5,catboost_legacy_comp_type_quota_spacetime_splines,Xpt_spatiotemporal__comp_type_quota__st_spacet...,catboost,MAE
6,catboost_legacy_comp_type_quota_geo_context,Xpt_spatiotemporal__comp_type_quota__st_geo_co...,catboost,MAE
7,catboost_legacy_comp_type_quota_spacetime_spli...,Xpt_spatiotemporal__comp_type_quota__st_spacet...,catboost,MAE
8,catboost_selected_binary_base_none,Xpt_spatiotemporal__selected_binary,catboost,MAE
9,catboost_selected_binary_base_spacetime_splines,Xpt_spatiotemporal__selected_binary__st_spacet...,catboost,MAE



[1/22] catboost_legacy_base_none

[2/22] catboost_legacy_base_spacetime_splines

[3/22] catboost_legacy_base_geo_context

[4/22] catboost_legacy_base_spacetime_splines+geo_context

[5/22] catboost_legacy_comp_type_quota_none

[6/22] catboost_legacy_comp_type_quota_spacetime_splines

[7/22] catboost_legacy_comp_type_quota_geo_context

[8/22] catboost_legacy_comp_type_quota_spacetime_splines+geo_context

[9/22] catboost_selected_binary_base_none

[10/22] catboost_selected_binary_base_spacetime_splines

[11/22] catboost_selected_binary_base_geo_context

[12/22] catboost_selected_binary_base_spacetime_splines+geo_context

[13/22] catboost_selected_binary_comp_type_quota_none

[14/22] catboost_selected_binary_comp_type_quota_spacetime_splines

[15/22] catboost_selected_binary_comp_type_quota_geo_context

[16/22] catboost_selected_binary_comp_type_quota_spacetime_splines+geo_context

[17/22] catboost_full_binary_base_spacetime_splines+geo_context

[18/22] catboost_full_binary_comp_type_quot

,arm,variant,homes,mdape,mape,p90_ape,delta_mdape_pp,delta_mape_pp,ci_low,ci_high,fraction_homes_improved,log_error_correlation
0,model,catboost_selected_binary_base_spacetime_spline...,1050,4.6685,6.1159,13.0367,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,model,catboost_selected_binary_base_none,1050,4.6941,6.1421,13.2532,0.0256,0.0262,-0.1208,0.2021,0.4800,0.9942
2,model,catboost_selected_binary_comp_type_quota_space...,1050,4.7131,6.1191,12.9986,0.0446,0.0032,-0.0888,0.2881,0.4971,0.9927
3,model,catboost_full_binary_base_spacetime_splines+ge...,1050,4.7141,6.1070,13.3121,0.0456,-0.0089,-0.1449,0.2051,0.4943,0.9945
4,model,catboost_selected_binary_base_spacetime_splines,1050,4.7161,6.1304,13.1650,0.0476,0.0145,-0.1274,0.2166,0.4876,0.9958
5,model,catboost_legacy_base_spacetime_splines+geo_con...,1050,4.7193,6.1347,13.1605,0.0508,0.0188,-0.1185,0.2342,0.4857,0.9951
6,model,catboost_selected_binary_comp_type_quota_space...,1050,4.7446,6.1052,13.1779,0.0761,-0.0107,-0.1091,0.2568,0.4962,0.9932
7,model,catboost_selected_binary_base_geo_context,1050,4.7545,6.1561,13.1506,0.0860,0.0402,-0.0951,0.2587,0.5000,0.9947
8,model,catboost_selected_binary_comp_type_quota_space...,1050,4.7601,6.1247,13.0066,0.0916,0.0088,-0.0520,0.3485,0.4743,0.9924
9,model,catboost_legacy_comp_type_quota_spacetime_splines,1050,4.7804,6.1387,12.9541,0.1119,0.0229,-0.1020,0.3541,0.5038,0.9919



MONTHLY MdAPE


month,2025-03,2025-04,2025-05
variant,,,
catboost_full_binary_base_spacetime_splines+geo_context,4.6457,4.6123,4.8417
catboost_full_binary_comp_type_quota_spacetime_splines+geo_context,4.8478,4.7477,4.9684
catboost_legacy_base_geo_context,4.9472,4.6307,5.1906
catboost_legacy_base_none,4.8397,4.8845,4.9953
catboost_legacy_base_spacetime_splines,4.6566,4.6392,4.9118
catboost_legacy_base_spacetime_splines+geo_context,4.5257,4.6171,4.9200
catboost_legacy_comp_type_quota_geo_context,4.8735,4.9629,5.0161
catboost_legacy_comp_type_quota_none,4.8376,4.8703,4.8819
catboost_legacy_comp_type_quota_spacetime_splines,4.8645,4.5709,4.8017



Saved to: C:\Users\deant\Desktop\Coding\Data Dash Case Comp\outputs\listaware_composite_catboost_mae


## 7 · Freeze a capped individual-model shortlist using ONLY the screen

The selection rule combines 65% MdAPE and 35% MAPE, ensures CatBoost/XGBoost/LightGBM representation, and includes the old list-aware controls. For the `curated` suite, additional feature anchors (selected-binary spatial, full-binary spatial, XGB splines, LGB splines and LGB advanced) are also retained. It reads **no** June–July labels.

You can inspect the proposed shortlist here. The confirmation runner reconstructs it from the saved screen results and stores their SHA-256 fingerprints so the selection cannot change silently after confirmation.


In [8]:
shortlist = None
if screen_pooled is not None:
    from models.listaware_candidate_pool import CONFIRMATION_ANCHORS
    shortlist = choose_confirmation(
        screen_pooled, top_k=TOP_K,
        anchors=CONFIRMATION_ANCHORS if SUITE == "curated" else (),
    )
    display(catalog_frame(shortlist))
    print("Confirming", len(shortlist), "candidates on FULL history.")
else:
    print("A completed pilot screen is required before choosing confirmation candidates.")


,variant,family,loss,target,features,tuned_parameters
0,cat_huber_baseline,catboost,Huber:delta=0.05,ratio,Xpt_spatiotemporal,False
1,cat_mae,catboost,MAE,ratio,Xpt_spatiotemporal,False
2,xgb_spline_geo,xgboost,RMSE,ratio,Xpt_spatiotemporal__selected_binary__st_spacet...,True
3,lgb_spline_geo,lightgbm,RMSE,ratio,Xpt_spatiotemporal__selected_binary__st_spacet...,True
4,cat_mae_spatial_geo,catboost,MAE,ratio,Xpt_spatiotemporal__selected_binary__st_spacet...,False
5,xgb_splines,xgboost,RMSE,ratio,Xpt_spatiotemporal__selected_binary__st_spacet...,True
6,lgb_splines,lightgbm,RMSE,ratio,Xpt_spatiotemporal__selected_binary__st_spacet...,True
7,cat_mae_binary,catboost,MAE,ratio,Xpt_spatiotemporal__selected_binary,False


Confirming 8 candidates on FULL history.


## 8 · Full-strength chronological confirmation (June–July)

**Only enable `RUN_CONFIRM` after you are satisfied with the stage-A shortlist.** Confirmation refits selected candidates monthly, on every eligible completed sale before that month, with the original complete iteration counts and all held-out homes.

Its validation months are chronologically later than the screening months. They are *not* a never-inspected holdout for the project, so report uncertainty accordingly. Candidate selection and hyperparameters are not adjusted based on these confirmation results.


In [9]:
confirm_pooled = confirm_monthly = confirm_types = confirm_predictions = None

if RUN_CONFIRM:
    if shortlist is None:
        raise RuntimeError("Finish and load stage-A screen scores first.")
    confirm_pooled, confirm_monthly, confirm_types, confirm_predictions = run(
        stage="confirm", suite=SUITE,
        screen_dir=SCREEN_DIR, output_dir=CONFIRM_DIR, top_k=TOP_K,
        threads=THREADS, seed=SEED, bootstrap_repeats=BOOTSTRAP_REPEATS,
    )
elif (CONFIRM_DIR / "pooled_scores.csv").is_file():
    confirm_record = json.loads((CONFIRM_DIR / "manifest.json").read_text())
    if confirm_record.get("stage") != "confirm" or tuple(
        confirm_record.get("months", ())
    ) != CONFIRM_MONTHS:
        raise ValueError("Confirmation output does not match expected months/stage.")
    confirm_pooled = pd.read_csv(CONFIRM_DIR / "pooled_scores.csv")
    confirm_monthly = pd.read_csv(CONFIRM_DIR / "monthly_scores.csv")
    print("Loaded full-strength confirmation results; no training.")
else:
    print("No completed confirmation; set RUN_CONFIRM=True when ready.")


[confirm 2025-06 1/8] cat_huber_baseline: fit on 127,762; predict 2,283
[confirm 2025-06 2/8] cat_mae: fit on 127,762; predict 2,283
[confirm 2025-06 3/8] xgb_spline_geo: fit on 127,762; predict 2,283
[confirm 2025-06 4/8] lgb_spline_geo: fit on 127,762; predict 2,283


KeyboardInterrupt: 

## 9 · Compare pilot and full-strength rankings (not raw accuracy)

Pilot and confirmation differ in **months, training-row counts, evaluation-row counts, and boosting rounds**, so comparing their absolute MdAPE as an apparent gain is misleading. Instead inspect whether the candidates' **relative ranking and within-fold month stability** persist under the stronger protocol.

Beware pilot-to-full reversal: we have already observed that in this project.


In [ ]:
if screen_pooled is not None and confirm_pooled is not None:
    screen_view = screen_pooled[["variant", "mdape", "mape"]].rename(
        columns={"mdape": "screen_mdape", "mape": "screen_mape"}
    )
    full_view = confirm_pooled[["variant", "mdape", "mape", "p90_ape"]].rename(
        columns={"mdape": "confirm_mdape", "mape": "confirm_mape"}
    )
    compared = full_view.merge(screen_view, on="variant", how="left",
                               validate="one_to_one")
    compared["screen_rank"] = compared.screen_mdape.rank(method="min")
    compared["confirm_rank"] = compared.confirm_mdape.rank(method="min")
    display(compared.sort_values("confirm_rank").round(4))
else:
    print("Both screen and confirmation scoreboards are needed.")


## 10 · What follows Stage A

**Do not freeze list-aware features after this screen.** Use the confirmed strong configurations, along with a few error-diverse alternatives, as candidates for:

1. **List-aware feature bagging:** selected vs additional BoW terms, weighted/random feature subsets, and multiple seeds.
2. **Row bootstrap/subbagging:** fit comparable-sale context and text vocabulary on legally eligible **full** historical training folds *before* sampling estimator rows.
3. **Model-specific Optuna:** tune CatBoost, XGBoost, and LightGBM on earlier chronological folds.
4. **Separate list-aware ensemble factory:** choose membership and weights based on marginal gain on separate selection folds and a frozen later-period audit.

The off-market factory remains independent, and the existing 50/35/15 list-aware blend remains a useful benchmark. No submission or saved production model is changed by this notebook.
